In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content/drive/MyDrive/tinyllama_qlora

!pwd

!ls


/content/drive/MyDrive/tinyllama_qlora
/content/drive/MyDrive/tinyllama_qlora
notebooks  requirements.txt  src


In [4]:
# !pip uninstall -y -r requirements.txt

!pip install -r requirements.txt

In [5]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("PEFT         :", peft.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)
print("BitsAndBytes :", bitsandbytes.__version__)

Transformers : 4.51.0
Datasets     : 3.6.0
PEFT         : 0.16.0
TRL          : 0.19.1
Accelerate   : 1.8.1
BitsAndBytes : 0.46.1


In [6]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [ ]:
from src.data.loader import load_data

DATASET_NAME = "databricks/databricks-dolly-15k"

dataset = load_data(DATASET_NAME)

print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [8]:
from src.data.to_chat_format import format_chat

new_data_set = dataset.map(format_chat)

print(new_data_set[0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'message': [{'content': "When did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced se

In [9]:
from src.models.tokenizer_loader import load_tokenizer
from src.data.apply_template import chat_template

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = load_tokenizer(MODEL_NAME)

templated_data = new_data_set.map(
    lambda x: chat_template(x, tokenizer)
)

print(templated_data[0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa', 'message': [{'content': "When did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced se

In [11]:
from src.data.filter_data import remove

train_data = remove(templated_data)

print(train_data[0])

{'text': "<|user|>\nWhen did Virgin Australia start operating?\n\nContext:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.</s>\n<|assistant|>\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.</s>\n"}
